In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (HemoDL)

This notebook curates the **HemoDL** dataset by integrating multiple FASTA files distributed across several folders (D1–D4). Peptide sequences are parsed from FASTA files, binary labels are inferred from filename conventions, duplicate consistency checks are applied, and the final curated dataset and metadata are exported for downstream analysis.

- **Toxic effect / endpoint:** hemolytic
- **Source:** HemoDL
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads FASTA files** (`.fa`) from the `D1`, `D2`, `D3`, and `D4` folders and concatenates them into a single table.
- **Infers hemolysis labels from filenames**:
  - filenames containing `"neg"` (case-insensitive) → `label = 0`,
  - all other files are treated as positive (`label = 1`).
- **Keeps a standardized schema**:
  - `sequence`
  - `label`
- **Checks duplicated sequences by sequence**:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv`,
  - `detected_error_sequences.csv`,
  - `metadata.json`.

In [2]:
name_source = "HemoDL"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
folders = ["D1", "D2", "D3", "D4"]
dfs = []

for folder in folders:
    for file in (Path(PATH_INPUT) / name_source / folder).glob("*.fa"):
        df = read_fasta_doc(file)
        df["source_file"] = file.name
        dfs.append(df)

df_HemoDL = pd.concat(dfs, ignore_index=True)

In [4]:
df_HemoDL = (
    df_HemoDL
    .assign(
        label=lambda d: d["source_file"]
            .str.contains("neg", case=False, na=False)
            .map({True: 0, False: 1})
    )
    [["sequence", "label"]]
)
df_HemoDL.shape

(10714, 2)

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_HemoDL, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_full.shape

(5437, 2)

In [7]:
df_errors.shape

(195, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_HemoDL)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2023, 4, 18, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from another DB',
 'repository or server': 'https://github.com/abcair/HemoDL/tree/main/data',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S0003269724000678',
 'number_of_raw_sequences': 10714,
 'number_of_sequences_retained': 5437,
 'number_of_positive_sequences': 2369,
 'number_of_negative_sequences': 3068,
 'number_of_erroneous_sequences': 195,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)